In [1]:
from profsea.emulator import GMSLREmulator

#### Introduction
The aim of this notebook is to break down the steps required to run the emulator version of ProFSea.

All of this can't run in Jupyter Notebook but the steps should be clear enough to follow along!

> If you want to use the CMIP6 patterns sterodynamic patterns instead of CMIP5, you need to run [this recipe](https://github.com/ESMValGroup/ESMValTool/blob/steric_patterns/esmvaltool/recipes/recipe_steric_patterns.yml) in ESMValTool. I recommend running with as much memory and processors as possible. 

> However, if you prefer not to go via ESMValTool feel free to reach out at <a href="mailto:gregory.munday\@dtc.ox.ac.uk">gregory.munday\@dtc.ox.ac.uk</a> and I can just send them over (they're not large in terms of data size).
---
All other data required for the tool can be [downloaded here](https://catalogue.ceda.ac.uk/uuid/47c849b907414f3ab5e56ba9cf83e779/).

I like keeping all my data in a `data/` folder next to the `profsea/` folder, which looks like this:

&emsp; `data/` \
&emsp;&emsp; `cmip5/` \
&emsp;&emsp; `cmip6/` \
&emsp;&emsp; `gia_estimates/` \
&emsp;&emsp; `grd_fingerprints/` \
&emsp;&emsp; `monte_carlo_timeseries/` \
&emsp;&emsp; `uk_cmip_slope_coefficients/`



#### Setting up the environment
The next step is to set up the conda environment. You should use the `ProFSea-emu-env.yml` environment file, building it with 

```bash
conda env create -f ProFSea-emu-env.yml
```

and then activating with `conda activate profsea-emu`.

---

One final step here is to install ProFSea as an editable package, using

```bash 
pip install -e . 
```

while inside the highest level directory.

#### Setup FaIR output
Run FaIR for the scenarios you want to simulate and save out the `global surface air temperature` and `ocean heat content`. You'll also need to calculate the 2015-2100 cumulative CO<sub>2</sub> emissions and save it to a `.json` file. The CMIP6 cumulative emissions are included in the repo: 

```profsea/emulator/aux_data/cumulative_cmip6_emissions.json```

Here's some sample code to calculate it from scratch in FaIR. Perhaps this could be included in this branch at some point.

```python
# Output cumulative emissions from 2015 to 2100
cum_emissions_dict = {}
for scenario in hist_gcam_ext.scenario.unique():  # loop over your scenarios
        emissions = hist_gcam_ext.loc[
            (hist_gcam_ext['scenario'] == scenario) & (hist_gcam_ext['variable'] == 'CO2 FFI'), 
            '2015':'2100'
        ].to_numpy()[0]  # select anthro emissions
        cum_emissions = np.cumsum(emissions)
        cum_emissions = cum_emissions / 1000  # Convert to PgC
        print(f'Total cumulative carbon emissions from 2015 to 2100 for {scenario}: ', cum_emissions[-1])
        cum_emissions_dict[scenario] = cum_emissions[-1]
        # Save to json
        with open('ngfs_data/cumulative_scenario_emissions.json', 'w') as f:
            json.dump(cum_emissions_dict, f)
```

You need to do this because the current Antarctic Ice-Sheet dynamics component uses the relationship between cumulative emissions and sea-level contribution to calculate the lower and upper bounds of the projection. This was implemented to remove scenario-dependence from the GMSLR emulator, and should be redundant when the component is updated.



#### Running global &rarr; local ProFSea projections
Point your paths in the `user-settings-emu.yml` file to the right places, fill in your scenario names and you should be ready to go!

Run: 

```bash
python spatial_projections.py
```

The global projections will be saved to `data/runs/gmslr_output/`, while the spatialised projections should be saved to `data/runs/<dir-specified-in-settings.yml>/`

The way this is accomplished (so that it runs on a laptop) is by taking and saving 101 percentiles of the GMSLR module output for each component. This can be updated via the `output_percentiles` keyword in the `GMSLR` class, currently line 458 in `spatial_projections.py`.

You can also run the GMSLR module by itself:

```python
from profsea.emulator import GMSLREmulator

emulator = GMSLREmulator(
    T_change,  # your T_change anom from FaIR
    OHC_change,  # your OHC_change anom from FaIR
    'low-overshoot',  # your scenario name (str)
    2300,  #  projection end year
    palmer_method=palmer_method,  # recommended for runs beyond 2100
    input_ensemble=True,  # must be true if providing a complete FaIR ensemble
    output_percentiles=np.arange(101),  # list/array of percentiles to sample
    cum_emissions_total=1450)  # int, cumulative emissions in scenario from 2015-2100

#  run the projections (auto-parallel)
emulator.project()

# Save to desired folder with the name of the scenario
emulator.save_components(
    os.path.join(settings["baseoutdir"], 'gmslr_output'),  # output dir
    scenario)  # scenario name

# Lists the available components
emulator.list_components()

# You can also access the data from each component directly:
emulator.greenland  # returns greenland projection ensemble

```